# 28×28 手写数字识别：KNN 完整入门教程

本 Notebook 根据原始项目整理，保留 **KNN、k=3、20% 测试集、random_state=1、stratify=y** 的实验设置，并补充更完整的注释、评估与错误排查。

## 学完后你应该能够

1. 解释一张 28×28 图片为什么对应 784 个特征；
2. 区分特征 `X` 和标签 `y`；
3. 完成数据读取、可视化、归一化和数据集拆分；
4. 使用 KNN 训练模型并评估分类效果；
5. 保存模型，并识别一张自己准备的 28×28 图片。

> **运行前准备：** 请先将训练文件 `手写数字识别.csv` 放入项目根目录下的 `data/` 文件夹。CSV 第一列必须是 0～9 的数字标签，后面必须恰好有 784 列像素值（范围 0～255）。本次上传的修改版 Notebook 没有包含训练 CSV。


## 目录与运行顺序

按顺序运行本 Notebook。推荐从项目根目录启动：

```bash
jupyter lab notebooks/01_手写数字识别_KNN完整教程.ipynb
```

项目内默认使用以下文件：

- `data/手写数字识别.csv`：训练数据（需自行放入）；
- `data/demo.png`：28×28 演示图片；
- `models/knn_digit_model.joblib`：训练完成后生成的模型文件。

### 环境依赖

在项目根目录执行 `python -m pip install -r requirements.txt`。如果你只想在 Notebook 中临时安装，可运行下一格。


In [ ]:
# 如果尚未安装依赖，可以取消注释并执行这一行。
# %pip install numpy pandas matplotlib scikit-learn pillow joblib


## 1. 导入工具包

- `pandas`：读取和处理 CSV 表格；
- `numpy`：处理图片像素矩阵；
- `matplotlib`：显示灰度图片、绘制评估图；
- `train_test_split`：拆分训练集与测试集；
- `KNeighborsClassifier`：K 近邻分类算法；
- `accuracy_score` 等：评估分类结果；
- `joblib`：保存和加载训练好的模型；
- `PIL.Image`：统一读取图片，得到灰度像素。


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 固定图片显示尺寸，28×28 = 784 个像素特征
IMAGE_WIDTH = 28
IMAGE_HEIGHT = 28
N_FEATURES = IMAGE_WIDTH * IMAGE_HEIGHT

# 自动定位项目根目录：既支持从项目根目录启动，也支持从 notebooks/ 启动
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "没有找到项目 data 文件夹。请从项目根目录启动 Jupyter，"
        "或确认 Notebook 所在目录结构。"
    )

DATA_PATH = PROJECT_ROOT / "data" / "手写数字识别.csv"
DEMO_IMAGE_PATH = PROJECT_ROOT / "data" / "demo.png"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "knn_digit_model.joblib"

print("项目根目录：", PROJECT_ROOT)
print("训练数据路径：", DATA_PATH)
print("演示图片路径：", DEMO_IMAGE_PATH)


## 2. 读取数据集并检查结构

当前案例沿用原始 Notebook 的数据组织方式：**第一列是标签，后面 784 列是像素特征**。

首先看数据行列数、前几行和标签分布。数据检查的目的是尽早发现列数不正确、标签范围不对、缺失值或像素范围异常等问题。


In [ ]:
# 读取 CSV 文件
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"找不到 {DATA_PATH}。请先把手写数字识别.csv 放入项目 data 文件夹。"
    )

df = pd.read_csv(DATA_PATH)

print("数据集形状（行数, 列数）：", df.shape)
print("前 5 行：")
display(df.head())

# 必须是 1 列标签 + 784 列像素
if df.shape[1] != N_FEATURES + 1:
    raise ValueError(
        f"数据应有 {N_FEATURES + 1} 列，当前为 {df.shape[1]} 列。"
        "请检查是否额外保存了索引列。"
    )

# 拆分标签 y 与特征 X
y = pd.to_numeric(df.iloc[:, 0], errors="raise").astype(int)
X_raw = df.iloc[:, 1:].apply(pd.to_numeric, errors="raise")
X_raw.columns = X_raw.columns.astype(str)  # sklearn 特征列名统一为字符串

if X_raw.isna().any().any():
    raise ValueError("像素数据存在缺失值，请先处理。")
if not set(y.unique()).issubset(set(range(10))):
    raise ValueError("标签列应只包含 0～9。")

print("特征矩阵 X 的形状：", X_raw.shape)
print("标签向量 y 的形状：", y.shape)
print("原始像素最小值：", X_raw.to_numpy().min())
print("原始像素最大值：", X_raw.to_numpy().max())
print("标签类别：", sorted(y.unique()))


### 理解 `X` 和 `y`

- `X`：模型的输入。每一行对应一张图片，每一列对应一个像素，所以一共有 784 列。
- `y`：模型要学习预测的答案。每一行是对应图片的真实数字，范围为 0～9。

这是一种监督学习任务：训练样本同时提供图片特征和正确标签。


## 3. 可视化一张数据集图片

CSV 中一张图片原本是一行 784 个数。要显示成图片，需要把这 784 个数重新排列成 28 行、28 列。

`reshape(28, 28)` 不会创造新像素，只是改变数组的形状。`cmap="gray"` 表示使用灰度颜色映射。


In [ ]:
def show_digit(index):
    """显示数据集中指定索引的手写数字，并打印它的真实标签。"""
    if index < 0 or index >= len(X_raw):
        raise IndexError(f"索引越界：有效范围是 0～{len(X_raw) - 1}")

    # iloc 按行位置取出一张图片的 784 个像素
    pixels = X_raw.iloc[index].to_numpy(dtype=np.float32)
    digit_image = pixels.reshape(IMAGE_HEIGHT, IMAGE_WIDTH)

    print(f"样本索引：{index}")
    print(f"真实数字标签：{y.iloc[index]}")
    print(f"展开后的特征形状：{pixels.shape}")
    print(f"还原后的图片形状：{digit_image.shape}")

    plt.figure(figsize=(3, 3))
    plt.imshow(digit_image, cmap="gray")
    plt.title(f"True label: {y.iloc[index]}")
    plt.axis("off")
    plt.show()

# 与你原 Notebook 中 show_digit(9) 的目的相同：查看第 10 条样本
show_digit(9)


## 4. 数据预处理：归一化

常见 MNIST CSV 中每个像素值为 0～255。这里把所有像素统一除以 255：

\[x_{scaled} = x / 255\]

处理后取值范围变为 0～1。KNN 依赖样本距离，因此训练数据与待预测图片必须采用一致的数值尺度、像素顺序、图片尺寸和黑白方向。

**注意：** 归一化只作用于像素特征 `X`，不能对标签 `y` 除以 255。


In [ ]:
pixel_min = float(X_raw.to_numpy().min())
pixel_max = float(X_raw.to_numpy().max())
if pixel_min < 0 or pixel_max > 255:
    raise ValueError(f"预期原始像素范围为 0～255，实际为 {pixel_min}～{pixel_max}")

X = X_raw.astype("float32") / 255.0

print("归一化后最小值：", X.to_numpy().min())
print("归一化后最大值：", X.to_numpy().max())
print("归一化后数据类型：", X.dtypes.iloc[0])


## 5. 拆分训练集和测试集

模型不能只在训练数据上检验自己。我们把 80% 数据用于训练，保留 20% 数据作为测试集。

- `test_size=0.2`：测试集占 20%；
- `random_state=1`：固定随机种子，让拆分结果可复现；
- `stratify=y`：分层抽样，尽可能让训练集和测试集保留相近的数字类别比例。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1,
    stratify=y,
)

print("训练集特征：", X_train.shape)
print("测试集特征：", X_test.shape)
print("训练集标签：", y_train.shape)
print("测试集标签：", y_test.shape)

print("\n训练集各类别数量：")
print(y_train.value_counts().sort_index())
print("\n测试集各类别数量：")
print(y_test.value_counts().sort_index())


## 6. 创建并训练 KNN 模型

KNN（K 近邻）是一种基于实例的分类方法。预测新图片时，算法会计算它与训练样本之间的距离，找出最近的 K 个样本，再根据邻居标签投票。

本案例沿用原项目的 `n_neighbors=3`：即选择最近的 3 个训练样本参与投票。


In [ ]:
# 创建 KNN 分类器，设置最近邻数量 k=3
estimator = KNeighborsClassifier(n_neighbors=3)

# fit 表示使用训练集学习：X_train 是图片特征，y_train 是正确标签
estimator.fit(X_train, y_train)

print("KNN 模型训练完成")


## 7. 模型评估

先使用测试集预测，再将预测标签和真实标签比较。

- **准确率（Accuracy）**：预测正确的样本数 / 测试样本总数；
- **分类报告**：展示每个数字类别的 precision、recall、f1-score 和样本数量；
- **混淆矩阵**：行表示真实类别，列表示预测类别。主对角线上的数字越多，通常表示正确分类越多。


In [ ]:
# 在测试集上进行预测
y_pred = estimator.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"测试集准确率：{accuracy:.4%}")

# 查看每个数字类别的 precision、recall 和 f1-score
print("\n分类报告：")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred, labels=list(range(10)))
display_labels = [str(i) for i in range(10)]

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title("KNN Confusion Matrix")
plt.show()


### 如何理解混淆矩阵？

例如，第 3 行、第 5 列的数值表示：真实标签为 2 的图片中，有多少张被预测成 4（行列标签按 0～9 排列）。可以重点查看非对角线中数值较大的位置，判断哪些数字容易混淆。


## 8. 保存训练好的模型

训练完成后，不需要每次识别图片都重新训练。可以使用 `joblib.dump()` 把模型保存到文件，再通过 `joblib.load()` 加载。

模型文件是训练后生成的本地文件，默认放在 `models/`，本项目的 `.gitignore` 会忽略它。


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(estimator, MODEL_PATH)
print(f"模型保存成功：{MODEL_PATH}")


## 9. 读取自己的 28×28 图片

预测图片必须和训练图片采用一致的预处理方式。这里使用 Pillow：

1. `convert("L")`：转为单通道灰度图；
2. 检查图片大小必须为 28×28；
3. 转成 NumPy 数组，保留 0～255 像素范围；
4. 除以 255，和训练阶段一致；
5. 展开成 1×784。

如果使用白底黑字的自定义图片，而训练数据是黑底白字，需要先反转颜色。下面默认不反转，因为项目样例图采用黑底白字。


In [ ]:
if not DEMO_IMAGE_PATH.exists():
    raise FileNotFoundError(f"找不到演示图片：{DEMO_IMAGE_PATH}")

image = Image.open(DEMO_IMAGE_PATH).convert("L")
if image.size != (IMAGE_WIDTH, IMAGE_HEIGHT):
    raise ValueError(
        f"演示图片必须为 28×28，当前为 {image.width}×{image.height}"
    )

image_pixels = np.asarray(image, dtype=np.float32)
print("图片形状：", image_pixels.shape)
print("原始像素范围：", image_pixels.min(), "～", image_pixels.max())

# 与训练数据一样，像素值除以 255
image_scaled = image_pixels / 255.0

# 由 28×28 展开为 1×784，1 表示本次只预测一张图片
image_features = image_scaled.reshape(1, N_FEATURES)
print("送入模型的特征形状：", image_features.shape)

plt.figure(figsize=(3, 3))
plt.imshow(image, cmap="gray")
plt.title("Input image")
plt.axis("off")
plt.show()


## 10. 使用模型进行预测

训练时传入的是 Pandas DataFrame，因此模型可能记录了训练特征名称。为了避免 `X does not have valid feature names` 警告，预测时也创建一个列名和顺序一致的 DataFrame。

如果模型没有记录特征名，则退回使用 NumPy 数组预测。


In [ ]:
# 用训练时相同的 784 个特征列名构造 DataFrame
if hasattr(estimator, "feature_names_in_"):
    image_df = pd.DataFrame(
        image_features,
        columns=estimator.feature_names_in_
    )
    y_pre = estimator.predict(image_df)
else:
    y_pre = estimator.predict(image_features)

print(f"预测图片值为：{int(y_pre[0])}")


## 11. 预测错误时，按顺序检查

如果模型对自定义图片预测不正确，不要只看图片是否为 28×28。建议依次检查：

1. **像素数值范围**：CSV 训练数据是否为 0～255，训练时是否除以 255；预测图片是否执行相同缩放；
2. **黑白方向**：训练数据是否黑底白字，图片是否白底黑字；
3. **尺寸与比例**：是否确实为 28×28，数字有没有被压扁或拉长；
4. **位置**：数字是否偏到边缘，训练数据和自写图片的居中方式是否差异很大；
5. **笔画粗细**：自写数字是否过细、过粗或存在噪点；
6. **标签与列顺序**：CSV 第一列是不是标签，后面的 784 列是不是按正确顺序展开的像素。

### 一个容易混淆的细节

`plt.imread()` 读取 PNG 时常返回 0～1 的浮点数；而 Pillow 的灰度图数组通常是 0～255 的整数。两种读取方式都可以，但归一化只能做一次：若数据已经是 0～1，就不要再次除以 255。当前 Notebook 使用 Pillow 读取，再显式 `/255`，让逻辑更清晰。


## 12. 复习题与扩展练习

请尝试自己回答并运行：

1. 为什么一张 28×28 图片会变成 784 个特征？
2. `X = X / 255` 改变的是图片标签还是像素特征？为什么？
3. `n_neighbors=3` 中的 3 表示什么？如果改成 5，预测结果可能发生什么变化？
4. 为什么不能用训练集准确率代替测试集准确率？
5. 如果图片是白底黑字，为什么可能需要反色？

**动手练习：** 将 `n_neighbors=3` 改成 1、5、7、9，分别记录测试集准确率；比较结果，并观察混淆矩阵中哪些数字最容易被混淆。

---

**完成标志：** 你已经能够独立完成 CSV 读取、图片可视化、KNN 训练、测试集评估、模型保存和自定义图片预测。
